<a href="https://colab.research.google.com/github/imnawar/thesis_repo/blob/main/Aljazeera_scraping_content_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Librraries

In [1]:
!pip install requests beautifulsoup4

# Import Librraries

In [2]:
import requests
from bs4 import BeautifulSoup
import os
from urllib.parse import urljoin, urlparse, parse_qs
from PIL import Image
from io import BytesIO

# Download the news

## Aljazeera

News urls

In [12]:
# 🔹 Add as many pages as you want here
URLS = [
    "https://www.aljazeera.net/ebusiness/2026/2/17/%D8%A3%D9%88%D8%B1%D9%88%D8%A8%D8%A7-%D8%AA%D8%AF%D8%B9%D9%85-%D8%AA%D8%B5%D9%86%D9%8A%D8%B9-%D8%A7%D9%84%D8%B3%D9%8A%D8%A7%D8%B1%D8%A7%D8%AA",
    "https://www.aljazeera.net/climate/2026/2/17/%D8%A3%D8%B3%D8%B1%D8%B9-%D9%82%D8%A7%D8%B1%D8%A7%D8%AA-%D8%A7%D9%84%D8%B9%D8%A7%D9%84%D9%85-%D8%A7%D8%AD%D8%AA%D8%B1%D8%A7%D8%B1%D8%A7-%D8%AF%D8%B9%D9%88%D8%A9-%D9%84%D8%A8%D9%86%D8%A7%D8%A1",
    "https://www.aljazeera.net/news/2026/2/17/%D8%AD%D8%A7%D9%85%D9%84%D8%A9-%D8%A7%D9%84%D8%B7%D8%A7%D8%A6%D8%B1%D8%A7%D8%AA-%D8%A7%D9%84%D9%81%D8%B6%D8%A7%D8%A6%D9%8A%D8%A9-%D8%A7%D9%84%D8%B5%D9%8A%D9%86%D9%8A%D8%A9-2",
    "https://www.aljazeera.net/tech/2026/2/17/%D8%A3%D9%85%D8%B1%D9%8A%D9%83%D8%A7-%D8%AA%D9%86%D9%82%D9%84-%D9%85%D9%81%D8%A7%D8%B9%D9%84%D8%A7-%D9%86%D9%88%D9%88%D9%8A%D8%A7-%D9%85%D8%B5%D8%BA%D8%B1%D8%A7-%D8%A8%D8%B7%D8%A7%D8%A6%D8%B1%D8%A9",
    "https://www.aljazeera.net/ebusiness/2026/2/17/%D9%85%D9%86-%D9%86%D9%8A%D8%AC%D9%8A%D8%B1%D9%8A%D8%A7-%D8%A5%D9%84%D9%89-%D8%A3%D9%88%D8%B1%D9%88%D8%A8%D8%A7-%D8%B9%D8%A8%D8%B1-%D8%A7%D9%84%D8%AC%D8%B2%D8%A7%D8%A6%D8%B1-%D8%AA%D8%B9%D8%B1%D9%81",
    "https://www.aljazeera.net/misc/2026/2/17/%D8%B9%D8%A7%D8%AC%D9%84-%D8%A7%D9%84%D8%B3%D8%B9%D9%88%D8%AF%D9%8A%D8%A9-%D9%88%D9%82%D8%B7%D8%B1-%D8%AA%D8%B9%D9%84%D9%86%D8%A7%D9%86-%D8%A3%D9%86-%D8%BA%D8%AF%D8%A7"
    # Add more URLs here
]

dirs settings

In [13]:
main_folder = "scraped_content"
source_folder = os.path.join(main_folder, "aljazeera")
page_folder = os.path.join(source_folder, "1")

Image download settings

In [14]:
MIN_IMAGE_AREA = 10000

def get_original_image_url(url):
    parsed = urlparse(url)
    params = parse_qs(parsed.query)

    if 'url' in params:
        return params['url'][0]

    clean_query = "&".join(
        f"{k}={v[0]}" for k, v in params.items()
        if k.lower() not in ['w', 'q', 'f']
    )

    return parsed._replace(query=clean_query).geturl()

def download_and_convert_image(image_url, folder, index):
    try:
        response = requests.get(image_url, timeout=10)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content)).convert("RGB")
        width, height = image.size

        if width * height < MIN_IMAGE_AREA:
            return None

        filename = f"{index}.jpg"
        path = os.path.join(folder, filename)
        image.save(path, "JPEG", quality=95)

        return {
            "filename": filename,
            "url": image_url,
            "area": width * height
        }

    except Exception:
        return None

Script for download single page

In [15]:
def scrape_single_page(url, page_number):
    print(f"\n🔹 Scraping page {page_number}")

    main_folder = source_folder
    page_folder = os.path.join(main_folder, str(page_number))
    images_folder = os.path.join(page_folder, "images")

    os.makedirs(images_folder, exist_ok=True)

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except Exception as e:
        print(f"Failed to fetch page {url}: {e}")
        return

    soup = BeautifulSoup(response.text, "html.parser")

    text_path = os.path.join(page_folder, "content.txt")
    image_index = 1
    image_metadata = []

    with open(text_path, "w", encoding="utf-8") as f:
        for element in soup.find_all(['h1','h2','h3','h4','h5','h6','p','img']):

            if element.name in ['h1','h2','h3','h4','h5','h6','p']:
                text = element.get_text(strip=True)
                if text:
                    f.write(f"[{element.name}] {text}\n\n")

            elif element.name == "img":
                src = element.get("src")
                if not src:
                    continue

                full_url = urljoin(url, src)
                original_url = get_original_image_url(full_url)

                result = download_and_convert_image(
                    original_url, images_folder, image_index
                )

                if result:
                    f.write(f"[img] {result['filename']}\n")
                    f.write(f"[img_src] {result['url']}\n\n")

                    image_metadata.append(result)
                    image_index += 1

    # Detect main image
    if image_metadata:
        main_image = max(image_metadata, key=lambda x: x["area"])
        os.rename(
            os.path.join(images_folder, main_image["filename"]),
            os.path.join(images_folder, "main_image.jpg")
        )


loop for multiple pages

In [16]:
def scrape_multiple_pages(url_list):
    for idx, url in enumerate(url_list, start=1):
        scrape_single_page(url, idx)

**MAIN**

In [17]:
scrape_multiple_pages(URLS)


🔹 Scraping page 1

🔹 Scraping page 2

🔹 Scraping page 3

🔹 Scraping page 4

🔹 Scraping page 5

🔹 Scraping page 6
